# Case-03: 柱子(垂直)方向幾何驗證

**專案**: pyfem-plastic-hinge（延續 Case-01/02）

**前置**: Case-02 已通過（RotSpring2D + BeamNL 組合，P=10N 時相對誤差 1.363e-08）。

**目標**：跟 Case-02 完全同一個力學模型，只是把梁從水平（x方向）換成垂直
（y方向，柱子），側向力也跟著換成水平方向——這是 portal frame 實際會用到
的柱子方向。對應 [[calculix-hinge2]] 那次「portal-frame-relevant geometry
test」。

**目的**：確認 `BeamNL` 的座標轉換（`getT`，依節點座標算元素局部座標系）在
非水平方向也正確。`RotSpring2D` 因為只用轉角自由度、不涉及座標轉換，理論
上方向怎麼變都不受影響——這個測試同時驗證這件事。

**預期**：數值大小應該跟 Case-02 完全一樣，只是方向換成側向。


In [ ]:
# ===== 0: 安裝 pyFEM（沿用 Case-01/02 的環境偵測邏輯，已安裝則略過）=====
import os
if os.path.isdir("/content"):
    PYFEM_DIR = "/content/PyFEM"
else:
    PYFEM_DIR = os.path.join(os.getcwd(), "PyFEM")

if not os.path.isdir(PYFEM_DIR):
    !git clone -q https://github.com/jjcremmers/PyFEM.git {PYFEM_DIR}
    %pip install -q -e {PYFEM_DIR} --break-system-packages
else:
    print(f"{PYFEM_DIR} 已存在，略過安裝")


## 1. 寫入 RotSpring2D（跟 Case-01/02 完全相同）

In [ ]:
rotspring_code = r'''
from pyfem.elements.Element import Element
from numpy import zeros


class RotSpring2D(Element):
    """2-node 2D 轉角彈簧元素 (pyFEM 自訂元素, Stage 1: 線性彈性)"""

    dofTypes = ['u', 'v', 'rz']

    def __init__(self, elnodes, props):
        Element.__init__(self, elnodes, props)
        self.family = "BEAM"

    def getTangentStiffness(self, elemdat):
        k = elemdat.props.k
        theta1 = elemdat.state[2]
        theta2 = elemdat.state[5]
        dtheta = theta2 - theta1
        M = k * dtheta

        elemdat.fint = zeros(6)
        elemdat.fint[2] = -M
        elemdat.fint[5] = M

        elemdat.stiff = zeros((6, 6))
        elemdat.stiff[2, 2] = k
        elemdat.stiff[2, 5] = -k
        elemdat.stiff[5, 2] = -k
        elemdat.stiff[5, 5] = k

    def getInternalForce(self, elemdat):
        self.getTangentStiffness(elemdat)
'''

target = f"{PYFEM_DIR}/pyfem/elements/RotSpring2D.py"
with open(target, "w") as f:
    f.write(rotspring_code)
print(f"已寫入 {target}")


## 2. Case-03 主測試：柱子(垂直)方向

跟 Case-02 唯一的差異：節點C座標從 `[L, 0.0]` 換成 `[0.0, L]`，施力方向從
垂直（v）換成水平（u）。其餘力學設定完全相同。


In [ ]:
import sys
sys.path.insert(0, PYFEM_DIR)

from pyfem.util.dataStructures import Properties, GlobalData
from pyfem.fem.NodeSet import NodeSet
from pyfem.fem.ElementSet import ElementSet
from pyfem.fem.DofSpace import DofSpace
from pyfem.fem.Assembly import assembleTangentStiffness
from pyfem.models.ModelManager import ModelManager
from numpy import zeros

E = 2.0e5; A = 1.0e4; I = 1.0e6; G = E / 2.6
L = 2000.0
k = 3.0e8
P = 10.0

def build_and_solve(P_applied):
    props = Properties()
    props.HingeElem = Properties({'type': 'RotSpring2D', 'k': k})
    props.BeamElem = Properties({'type': 'BeamNL', 'E': E, 'A': A, 'I': I, 'G': G})

    nodes = NodeSet()
    nodes.add(1, [0.0, 0.0])   # A: 地面(柱底)
    nodes.add(2, [0.0, 0.0])   # B: 跟A重合，經彈簧連接
    nodes.add(3, [0.0, L])     # C: 柱頂(垂直方向，對比 Case-02 的 [L,0.0])

    elements = ElementSet(nodes, props)
    elements.add(1, 'HingeElem', [1, 2])
    elements.add(2, 'BeamElem', [2, 3])

    dofs = DofSpace(elements)
    cons = dofs.createConstrainer()
    for dtype in ['u', 'v', 'rz']:
        cons.addConstraint(dofs.getForType(1, dtype), 0.0, "main")
    for dtype in ['u', 'v']:
        cons.addConstraint(dofs.getForType(2, dtype), 0.0, "main")
    cons.flush()

    globdat = GlobalData(nodes, elements, dofs)
    globdat.models = ModelManager(props, globdat)

    a = globdat.state
    fext = zeros(len(dofs))
    loadDof = dofs.getForType(3, 'u')   # 側向力(水平, FX)
    fext[loadDof] = P_applied

    for _ in range(20):
        K, fint = assembleTangentStiffness(props, globdat)
        r = fext - fint
        if dofs.norm(r) < 1e-6 * max(P_applied, 1.0):
            break
        da = dofs.solve(K, r)
        a[:] += da[:]
    else:
        raise RuntimeError("Newton-Raphson 未收斂")

    K, fint = assembleTangentStiffness(props, globdat)
    residual = dofs.norm(fext - fint)
    tipDof = dofs.getForType(3, 'u')
    return a[tipDof], residual

def hand_calc(P_applied):
    theta0 = (P_applied * L) / k
    delta_beam = P_applied * L**3 / (3 * E * I)
    return theta0 * L + delta_beam

delta_numeric, residual = build_and_solve(P)
delta_hand = hand_calc(P)
rel_err = abs(delta_numeric - delta_hand) / abs(delta_hand)

print("=== Case-03: 柱子(垂直)方向幾何驗證 ===")
print(f"施加側向力 P      = {P:.3f} N")
print(f"柱頂側向位移(手算) = {delta_hand:.6f} mm")
print(f"柱頂側向位移(數值) = {delta_numeric:.6f} mm")
print(f"相對誤差          = {rel_err:.3e}")
print(f"殘差(平衡自我檢查)  = {residual:.3e}")

# 跟 Case-02 的數值直接比對(方向不同，大小應該完全一樣)
delta_case02 = 0.266667  # Case-02 在 P=10N 時的手算/數值結果(mm)
print(f"\n跟 Case-02(水平梁)比對: Case-02={delta_case02:.6f} mm, "
      f"Case-03(垂直柱)={abs(delta_numeric):.6f} mm, "
      f"差異={abs(abs(delta_numeric)-delta_case02):.3e} mm")

assert rel_err < 1e-5, "垂直方向跟手算不符，座標轉換可能有問題"
assert residual < 1e-6, "殘差未收斂"
assert abs(abs(delta_numeric) - delta_case02) < 1e-4, "跟 Case-02 數值對不起來，方向轉換有問題"
print("\n✅ PASS")


## 3. 結論與下一步

實際在這個沙盒環境跑過（不是預期結果）：相對誤差 `1.363e-08`，殘差
`2.668e-07`，跟 Case-02（水平梁）的數值差異只有 `3.370e-07` mm（浮點雜訊
等級）——確認 `BeamNL` 的座標轉換在垂直方向正確，`RotSpring2D` 因為只用
轉角自由度、不受方向影響，兩者組合在 portal frame 實際會用到的柱子方向上
是可信的。

**Validation Log**

| ID | 主題 | 比對對象 | 結果 |
|---|---|---|---|
| VL-01 | RotSpring2D 孤立線性行為 | 手算 M/k | 機器精度一致 (rel_err=0) |
| VL-02 | RotSpring2D+BeamNL 組合撓度(水平) | 手算疊加 | rel_err=1.363e-08 (P=10N) |
| VL-03 | RotSpring2D+BeamNL 組合撓度(垂直柱) | 手算疊加 + Case-02 數值比對 | rel_err=1.363e-08，跟Case-02差異3.37e-07mm |

**下一步（Case-04）**：加入非線性 M-θ（Mp 封頂），用 `self.history` 記錄
降伏狀態，並明確做一次「Newton-Raphson 疊代真的有處理到這個自訂元素的
非線性」自我檢查（對應 CalculiX 那次 `NLGEOM` 陷阱的教訓，這裡是必檢項
目，不是可有可無的步驟）。
